In [1]:
# Import packages
import pandas as pd
import numpy as np

from functions import *

In [2]:
# Read in and check data
d = pd.read_csv("~/ml/data/kaggle_cancer/lung_cancer_mortality_data_large.csv")

print(f"Dataset Size: {d.shape[0]} Columns, {d.shape[1]} Rows\n")

print(f"Column Names:")
for i in d.columns:
    print(i)

d.head()

Dataset Size: 3250000 Columns, 17 Rows

Column Names:
id
age
gender
country
diagnosis_date
cancer_stage
family_history
smoking_status
bmi
cholesterol_level
hypertension
asthma
cirrhosis
other_cancer
treatment_type
end_treatment_date
survived


,id,age,gender,country,diagnosis_date,cancer_stage,family_history,smoking_status,bmi,cholesterol_level,hypertension,asthma,cirrhosis,other_cancer,treatment_type,end_treatment_date,survived
0,1,64.0,Female,Germany,2016-04-05,Stage III,No,Never Smoked,31.1,257,1,1,0,0,Combined,2017-11-13,0
1,2,50.0,Male,Czech Republic,2023-04-20,Stage III,Yes,Passive Smoker,25.9,208,1,0,0,0,Radiation,2024-09-02,0
2,3,65.0,Male,Romania,2023-04-05,Stage IV,No,Never Smoked,18.9,193,0,0,0,0,Surgery,2024-10-08,0
3,4,51.0,Female,Latvia,2016-02-05,Stage III,Yes,Former Smoker,34.6,249,1,1,1,0,Surgery,2017-05-08,1
4,5,37.0,Male,Greece,2023-11-29,Stage I,Yes,Never Smoked,40.2,262,0,0,0,0,Chemotherapy,2025-05-03,0


In [3]:
# Define target
target = "survived"
d[target].value_counts() / len(d)

# Create Year/Month Column
d['year_month'] = d['end_treatment_date'].str[0:4] + "/" + d['end_treatment_date'].str[5:7]

In [ ]:
tab = aggregate_table(
    data=d, 
    factor='year_month', 
    columns=['id', target],
    aggs=['count', 'mean']
    )

fig = plotly_plot(
    x = tab['year_month'], 
    y_list = [tab['id'], tab[target]], 
    y_names = ['Count', 'Surivived'],
    secondary_y = [False, True], 
    plot_type = ['bar', 'scatter'], 
    colours = ['blue', 'red'],
    title = 'Survival Rate by End of Treatment Date',
    x_title = 'End of Treatment Date',
    y_title = 'Row Count',
    y2_title = 'Survival Rate',
    y_range = [0, 80000],
    y2_range = [0.18, 0.24],
    # x_range = ['2020-01-01', '2024-11-30']
    )

fig

In [5]:
# Create feature list and split into numerical and categorical types
features = [
    i,
    'gender',
    'country',
    'cancer_stage',
    'family_history',
    'smoking_status',
    'bmi',
    'cholesterol_level',
    'hypertension',
    'asthma',
    'cirrhosis',
    'other_cancer',
    'treatment_type',
]

d[features].dtypes

cats = [i for i in features if d[i].dtype == 'object']
nums = [i for i in features if i not in cats]

print("Numerical Features: ")
for i in nums:
    print(i)

print("\nCategorical Features: ")
for i in cats:
    print(i)

Numerical Features: 
survived
bmi
cholesterol_level
hypertension
asthma
cirrhosis
other_cancer

Categorical Features: 
gender
country
cancer_stage
family_history
smoking_status
treatment_type


In [6]:
# Summary statistics
display(numerical_summary(d, nums))
display(categorical_summary(d, cats))

,Total Rows,Missing Rows,Populated Rows,Unique Values,Maximum,Minimum,Range,Median,Upper Quartile,Lower Quartile,IQR,Variance,Std Deviation
survived,3250000.0,0.0,3250000.0,2.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.171376,0.413976
bmi,3250000.0,0.0,3250000.0,291.0,45.0,16.0,29.0,30.5,37.7,23.2,14.5,70.081911,8.371494
cholesterol_level,3250000.0,0.0,3250000.0,151.0,300.0,150.0,150.0,242.0,271.0,196.0,75.0,1887.773296,43.448513
hypertension,3250000.0,0.0,3250000.0,2.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,0.187581,0.433107
asthma,3250000.0,0.0,3250000.0,2.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.249022,0.499021
cirrhosis,3250000.0,0.0,3250000.0,2.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.175243,0.418621
other_cancer,3250000.0,0.0,3250000.0,2.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.080279,0.283335


NameError: name 'd' is not defined

In [ ]:
# Distributions by feature
for i in features:
    df = d.copy()

    if (i in nums) & (len(df[i].unique()) > 2):
        ignore, bands = pd.cut(df[i], bins=20, retbins=True)
        labels = [f"{bands[j]} to {bands[j+1]}" for j in range(len(bands)-1)]
        df[i] = pd.cut(df[i], bins=bands, labels = labels)

    tab = aggregate_table(
        data=df, 
        factor=i, 
        columns=['id'],
        aggs=['count']
        )

    fig = plotly_plot(
        x = tab[i], 
        y_list = [tab['id']], 
        y_names = ['Count'],
        secondary_y = [False], 
        plot_type = ['bar'], 
        colours = ['blue'],
        title = f'Row Count by {i}',
        x_title = i,
        y_title = 'Row Count',
        # y_range = [0, 80000],
        # y2_range = [0.18, 0.24],
        # x_range = ['2020-01-01', '2024-11-30']
        )

    display(fig)